# Preparacion de datos para segmentar lesiones de COVID en tomografias

## Equipo de trabajo

- Ximena Perez Escalante
- Jacob Espejel Guiza
- Sebastian Osorio Arteaga

**Notebook de referencia:** [PyTorch Baseline for Semantic Segmentation](https://www.kaggle.com/code/maedemaftouni/pytorch-baseline-for-semantic-segmentation)

## Objetivo del proyecto

Este notebook prepara tomografias de pulmon para una tarea de **segmentacion**. El objetivo
de la segmentacion es asignar una clase a cada pixel y asi delimitar las zonas asociadas con
lesiones pulmonares.

## Flujo que realizamos y por que

1. **Verificar los archivos:** confirma que Kaggle tiene disponible el conjunto correcto.
2. **Cargar imagenes y mascaras:** convierte los datos a formatos que consumen menos memoria.
3. **Revisar ejemplos:** comprueba que cada tomografia coincide con su mascara y permite
   reconocer visualmente las cuatro clases.
4. **Unificar el formato de las mascaras:** deja una sola etiqueta por pixel para facilitar
   su uso posterior.
5. **Homogeneizar las intensidades:** reduce el efecto de valores extremos y coloca todas
   las imagenes en una escala comparable.
6. **Comparar las dos fuentes:** permite reconocer si MedSeg y Radiopaedia presentan
   diferencias que puedan influir en el proyecto.
7. **Separar y reunir los datos:** reserva una parte para validacion y aprovecha el resto
   para entrenamiento.

## Que representan los datos

Cada imagen es un corte horizontal de una tomografia computarizada de torax. Los valores de
los pixeles se expresan en **unidades Hounsfield (HU)**, que representan densidad. Por eso no
deben interpretarse como el brillo de una fotografia comun: el aire se encuentra cerca de
-1000 HU, el agua alrededor de 0 HU y los tejidos densos tienen valores positivos.

Cada corte mide 512 x 512 pixeles y tiene una mascara del mismo tamano. La mascara funciona
como la respuesta esperada: indica a que region pertenece cada pixel.

## Significado de las cuatro clases

| Clase | Nombre | Que representa | Papel en el proyecto |
|---|---|---|---|
| 0 | Ground glass | Opacidad tenue o aspecto de vidrio esmerilado. El pulmon conserva parte del aire, pero presenta inflamacion. | Lesion de interes que debe localizarse. |
| 1 | Consolidation | Region mas densa en la que el aire de los alveolos ha sido reemplazado por liquido o material inflamatorio. | Lesion de interes que debe localizarse. |
| 2 | Lungs other | Tejido pulmonar que no fue marcado como alguna de las dos lesiones anteriores. No equivale necesariamente a pulmon sano. | Sirve como contexto anatomico. |
| 3 | Background | Todo lo que queda fuera del pulmon, como pared toracica, corazon, camilla o aire exterior. | Separa el pulmon del resto de la imagen. |

Las clases 0 y 1 son el objetivo clinico principal. Las clases 2 y 3 ayudan a que el sistema
distinga una lesion del tejido pulmonar restante y del fondo.

## Fuentes disponibles

| Archivos | Cortes | Uso |
|---|---:|---|
| `images_medseg.npy` y `masks_medseg.npy` | 100 | Imagenes etiquetadas de MedSeg. |
| `images_radiopedia.npy` y `masks_radiopedia.npy` | 829 | Imagenes etiquetadas de Radiopaedia. |
| `test_images_medseg.npy` | 10 | Imagenes sin mascara reservadas para una prediccion posterior. |

Se combinan MedSeg y Radiopaedia porque disponer de mas ejemplos puede mejorar el aprendizaje.
Sin embargo, provienen de fuentes diferentes y pueden reflejar distintos equipos o protocolos;
por eso se revisan antes de unirlas.

---
# 1. Preparar el entorno y verificar los archivos

Esta etapa carga las herramientas necesarias y muestra los archivos disponibles en Kaggle.
La lista funciona como una comprobacion inicial: si los cinco archivos `.npy` no aparecen,
el conjunto de datos no se agrego correctamente y las siguientes etapas no podran ejecutarse.

Archivos esperados:

- `images_medseg.npy`
- `masks_medseg.npy`
- `test_images_medseg.npy`
- `images_radiopedia.npy`
- `masks_radiopedia.npy`


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


---
# 2. Cargar las imagenes y las mascaras

En esta etapa se cargan los cinco archivos completos. Las imagenes se guardan como
`float32` y las mascaras como `int8` para reducir el uso de memoria sin perder informacion
necesaria para el proyecto.

La forma de cada arreglo ayuda a entender su contenido:

| Variable | Forma | Interpretacion |
|---|---|---|
| `images_radiopedia` | `(829, 512, 512, 1)` | 829 cortes en escala de grises. |
| `masks_radiopedia` | `(829, 512, 512, 4)` | Una capa por cada clase. |
| `images_medseg` | `(100, 512, 512, 1)` | 100 cortes en escala de grises. |
| `masks_medseg` | `(100, 512, 512, 4)` | Una capa por cada clase. |
| `test_images_medseg` | `(10, 512, 512, 1)` | 10 cortes sin mascara. |

La decision de cargar ambas fuentes desde el inicio permite aplicarles el mismo flujo. El
costo es un consumo alto de memoria, que se reduce usando tipos de datos mas pequenos y
liberando posteriormente los arreglos que dejan de ser necesarios.


In [ ]:
prefix = '/kaggle/input/competitions/covid-segmentation/'

images_radiopedia = np.load(os.path.join(prefix, 'images_radiopedia.npy')).astype(np.float32)
masks_radiopedia = np.load(os.path.join(prefix, 'masks_radiopedia.npy')).astype(np.int8)
images_medseg = np.load(os.path.join(prefix, 'images_medseg.npy')).astype(np.float32)
masks_medseg = np.load(os.path.join(prefix, 'masks_medseg.npy')).astype(np.int8)

test_images_medseg = np.load(os.path.join(prefix, 'test_images_medseg.npy')).astype(np.float32)

---
# 3. Crear una vista sencilla de las imagenes y sus mascaras

La siguiente celda prepara una visualizacion de apoyo. Su finalidad es mostrar varias
tomografias junto con las regiones marcadas en sus mascaras.

Esta revision permite confirmar dos aspectos antes de transformar los datos:

- que imagen y mascara correspondan al mismo corte;
- que las cuatro capas representen las clases en el orden esperado.


In [ ]:
def visualize(image_batch, mask_batch=None, pred_batch=None, num_samples=8, hot_encode=True):
    num_classes = mask_batch.shape[-1] if mask_batch is not None else 0
    fix, ax = plt.subplots(num_classes + 1, num_samples, figsize=(num_samples * 2, (num_classes + 1) * 2))

    for i in range(num_samples):
        ax_image = ax[0, i] if num_classes > 0 else ax[i]
        if hot_encode:
            ax_image.imshow(image_batch[i,:,:,0], cmap='Greys')
        else:
            ax_image.imshow(image_batch[i,:,:])
        ax_image.set_xticks([])
        ax_image.set_yticks([])

        if mask_batch is not None:
            for j in range(num_classes):
                if pred_batch is None:
                    mask_to_show = mask_batch[i,:,:,j]
                else:
                    mask_to_show = np.zeros(shape=(*mask_batch.shape[1:-1], 3))
                    mask_to_show[..., 0] = pred_batch[i,:,:,j] > 0.5
                    mask_to_show[..., 1] = mask_batch[i,:,:,j]
                ax[j + 1, i].imshow(mask_to_show, vmin=0, vmax=1)
                ax[j + 1, i].set_xticks([])
                ax[j + 1, i].set_yticks([])

    plt.tight_layout()
    plt.show()

---
# 4. Revisar como aparecen las clases en los cortes

Radiopaedia contiene volumenes completos: varios cortes consecutivos pueden pertenecer al
mismo estudio y ser muy parecidos entre si. Por tanto, no debe asumirse que cada corte
representa un paciente independiente.

En la visualizacion, la primera fila muestra la tomografia y las cuatro filas siguientes
muestran las clases por separado:

1. ground glass;
2. consolidation;
3. lungs other;
4. background.

Esta lectura visual ayuda a distinguir las lesiones del contexto anatomico. Tambien permite
anticipar que las regiones de lesion suelen ocupar menos espacio que el fondo, una condicion
que debera considerarse cuando se entrene y evalue un modelo.

Se muestran los cortes 30 a 37 porque forman un grupo consecutivo suficiente para observar
como cambia la anatomia dentro de un mismo volumen.


In [ ]:
visualize(images_radiopedia[30:], masks_radiopedia[30:])

---
# 5. Dejar una sola clase por pixel

Las mascaras originales guardan cuatro capas: una para cada clase. Para las siguientes etapas
se transforman en un mapa unico, donde cada pixel contiene directamente un valor entre 0 y 3.

Por ejemplo, si un pixel pertenece a consolidacion, su valor final es `1`. La misma regla se
aplica a las demas clases.

Esta decision tiene dos ventajas para el proyecto:

- hace mas directa la lectura de la etiqueta de cada pixel;
- reduce de cuatro capas a una, lo que disminuye el uso de memoria.

La visualizacion posterior muestra todas las clases reunidas en una sola mascara y permite
confirmar que la conversion conservo su ubicacion.


In [ ]:
def onehot_to_mask(mask, palette):
    """
    Converts a mask (H, W, K) to (H, W, C)
    """
    x = np.argmax(mask, axis=-1)
    colour_codes = np.array(palette)
    x = np.uint8(colour_codes[x.astype(np.uint8)])
    return x

palette = [[0], [1], [2],[3]]
masks_radiopedia_recover = onehot_to_mask(masks_radiopedia, palette).squeeze()  # shape = (H, W)

masks_medseg_recover = onehot_to_mask(masks_medseg, palette).squeeze()  # shape = (H, W)

print('Hot encoded mask size: ',masks_radiopedia.shape)
print('Paletted mask size:',masks_medseg_recover.shape)

visualize(masks_medseg_recover[30:],hot_encode=False)

---
# 6. Homogeneizar las intensidades de las tomografias

Las tomografias pueden incluir valores muy altos o muy bajos causados por hueso denso,
objetos metalicos, aire exterior o ruido de reconstruccion. Si esos extremos dominan la
escala, las diferencias sutiles dentro del pulmon resultan mas dificiles de aprovechar.

Por esa razon se toman dos decisiones:

1. **Limitar el intervalo a -1500 y 500 HU.** Este rango conserva la informacion relevante
   del torax y evita que unos pocos valores extremos condicionen todo el conjunto.
2. **Usar una escala comun.** Se calculan una media y una desviacion de referencia con
   Radiopaedia y se aplican tambien a MedSeg y al conjunto de prueba.

Para calcular la referencia se utiliza la zona central de la distribucion, entre los
percentiles 5 y 95. Asi, la escala describe mejor los tejidos y depende menos del aire que
rodea al cuerpo o de valores aislados.

Reutilizar la misma referencia es una decision esencial: todos los cortes quedan comparables
y el conjunto de prueba no participa en el calculo de los valores usados para prepararlo.
Esto evita introducir anticipadamente informacion del conjunto que despues se pretende
evaluar.


In [ ]:
def preprocess_images(images_arr, mean_std=None):
    images_arr[images_arr > 500] = 500
    images_arr[images_arr < -1500] = -1500
    min_perc, max_perc = np.percentile(images_arr, 5), np.percentile(images_arr, 95)
    images_arr_valid = images_arr[(images_arr > min_perc) & (images_arr < max_perc)]
    mean, std = (images_arr_valid.mean(), images_arr_valid.std()) if mean_std is None else mean_std
    images_arr = (images_arr - mean) / std
    print(f'mean {mean}, std {std}')
    return images_arr, (mean, std)

images_radiopedia, mean_std = preprocess_images(images_radiopedia)
images_medseg, _ = preprocess_images(images_medseg, mean_std)
test_images_medseg, _ = preprocess_images(test_images_medseg, mean_std)

---
# 7. Revisar la compatibilidad entre MedSeg y Radiopaedia

Aunque ambas fuentes contienen tomografias y usan las mismas clases, no necesariamente fueron
obtenidas con los mismos equipos o protocolos. El histograma compara sus intensidades despues
de aplicar la escala comun.

Si las curvas no coinciden por completo, significa que la procedencia de la imagen todavia
influye en sus valores. Aun asi, se decide combinar las fuentes porque aumentar de 100 a 929
cortes etiquetados aporta mucha mas variedad para un entrenamiento posterior.

La consecuencia de esta decision es que los resultados futuros deben interpretarse teniendo
en cuenta la fuente de los datos. Un sistema que aprende principalmente de Radiopaedia puede
comportarse de manera distinta sobre MedSeg, que es tambien la procedencia de las imagenes de
prueba.


In [ ]:
def plot_hists(images1, images2=None):
    plt.hist(images1.ravel(), bins=100, density=True, color='b', alpha=1 if images2 is None else 0.5)
    if images2 is not None:
        plt.hist(images2.ravel(), bins=100, density=True, alpha=0.5, color='orange')
    plt.show();

plot_hists(test_images_medseg, images_radiopedia)

---
# 8. Formar los conjuntos de entrenamiento y validacion

La ultima transformacion organiza los datos segun el uso que tendrian en una etapa posterior:

- **Validacion:** los primeros 24 cortes de MedSeg.
- **Entrenamiento:** los 76 cortes restantes de MedSeg y los 829 cortes de Radiopaedia.

El conjunto de entrenamiento queda con 905 cortes y el de validacion con 24. Radiopaedia se
incluye completa en entrenamiento para aprovechar su mayor cantidad de ejemplos, mientras
que MedSeg aporta tanto ejemplos de entrenamiento como una referencia de validacion cercana
a la fuente del conjunto de prueba.

Esta separacion reproduce la decision del proyecto original, pero tiene una limitacion:
los 24 cortes de validacion son consecutivos y pueden ser muy parecidos. Una continuacion
del proyecto deberia separar por paciente o por volumen completo, siempre que esos
identificadores esten disponibles, para medir el desempeno sobre estudios realmente distintos.

Al final se liberan los arreglos originales porque la informacion necesaria ya se encuentra
en los conjuntos finales. Esta medida evita mantener copias grandes en memoria.


In [ ]:
val_indexes = list(range(24))
train_indexes = list(range(24, 100))

train_images = np.concatenate((images_medseg[train_indexes], images_radiopedia))
train_masks = np.concatenate((masks_medseg_recover[train_indexes], masks_radiopedia_recover))
val_images = images_medseg[val_indexes]
val_masks = masks_medseg_recover[val_indexes]

del masks_medseg_recover
del masks_radiopedia_recover
del images_radiopedia
del masks_radiopedia
del images_medseg
del masks_medseg


---
# 9. Resultado del flujo

Los datos quedan preparados para una futura etapa de modelado:

- 905 cortes y sus mascaras para entrenamiento;
- 24 cortes y sus mascaras para validacion;
- 10 cortes de prueba preparados con la misma referencia de intensidad.

Las decisiones principales fueron conservar la informacion clinica de las tomografias,
reducir el consumo de memoria, representar una sola clase por pixel, aplicar una escala comun
y aprovechar las dos fuentes disponibles.

La principal precaucion para una siguiente version es mejorar la separacion de validacion.
Los cortes cercanos de una tomografia no son observaciones independientes; por ello, una
division por paciente o por volumen ofreceria una evaluacion mas confiable.

El entrenamiento, la arquitectura del modelo, las metricas y la generacion de predicciones
quedan fuera del alcance de este notebook.
